# Generic 1D PDE Solver

This notebook implements a high-performance, generic 1D partial differential equation (PDE) solver for equations of the form:

$$
\partial_t u = \hat{L}u + N(u) + S(x, t)
$$

defined on a periodic spatial domain $x \in [-L_x/2, L_x/2)$. The solver utilizes a split-step (Strang splitting) approach, leveraging the `psiop` library for pseudo-differential linear operators and spectral methods for nonlinearities.

Everything that defines a simulation is treated as an explicit input, grouped into four main categories:

### 1. Configuration & Inputs

*   **Problem Definition (`Problem`):**
    *   $\hat{L}$: Linear operator, represented as a SymPy symbol $L(x, \xi; \mathbf{p})$ in phase space.
    *   $N(u, x, t; \mathbf{p})$: Nonlinear term, written as a rate using SymPy.
    *   $S(x, t; \mathbf{p})$: Source term using SymPy.
    *   $\mathbf{p}$: A tuple of SymPy symbols (e.g., $\chi, A_{\text{src}}$) along with their default values.
    *   *Quantization:* `'weyl'` (default) or `'kohn-nirenberg'`.
    *   *`psiop` options:* Taylor order, application backend, and kwargs.
*   **Geometry (`Geometry`):** A periodic segment $[-L_x/2, L_x/2)$ discretized with $N_x$ collocation points.
*   **Numerics (`Numerics`):** Final integration time, number of time steps, saved frames, floating-point precision, and parallelization settings.
*   **Runs (`Run`):** A batch of simulations. Each `Run` specifies an initial condition $u(x,0)$ and a dictionary of parameter values. Parameter sweeps are handled by simply providing a list of `Run` instances with different parameter values.

### 2. Mathematical Conventions

*   The spatial domain is periodic, utilizing spectral (FFT) discretization.
*   Fourier modes are proportional to $e^{i k x}$, mapping the spatial derivative to $\partial_x \leftrightarrow i\xi$ in the symbol $L$.
    *   *Example:* The symbol $L = -i\xi^2$ corresponds to the physical operator $\partial_t u = i \partial_{xx} u$.
*   $N(u)$ is treated as a rate (e.g., $N = i \chi |u|^2 u$). It may contain spatial derivatives of $u$, complex conjugation, and explicit $(x, t)$ dependencies.
*   $S(x,t)$ is added directly to $\partial_t u$ without any hidden imaginary factors.

### 3. Numerical Scheme

#### Linear Evolution (Pseudo-Differential Propagator)
The linear component of the split-step scheme requires applying the evolution operator $e^{\Delta t \hat{L}}$ to the field $u(x,t)$, where $\hat{L}$ is the pseudo-differential operator defined by the symbol $L(x, \xi)$. 

Because exact exponentiation of general pseudo-differential operators is computationally intractable, the solver approximates the propagator using a truncated Taylor series of a specified `order`:
$$
e^{\Delta t \hat{L}} \approx \sum_{j=0}^{\text{order}} \frac{(\Delta t \hat{L})^j}{j!}
$$

For every distinct set of linear parameters, the solver executes the following procedure:
1. **Quantization Mapping:** If Weyl quantization is active, the symbol $L(x, \xi)$ is analytically converted to its Kohn-Nirenberg form $L_{\text{kn}}(x, \xi)$ to preserve correct operator ordering.
2. **Operator Expansion:** A propagator family is constructed from $L_{\text{kn}}$ using the specified Taylor `order`.
3. **Fractional Step Extraction:** Specific operators are extracted for the symmetric Strang splitting sequence:
   $$
   \hat{U}_{\text{half}} = \exp\left(+\frac{\Delta t}{2} \hat{L}\right), \quad \hat{U}_{\text{full}} = \exp\left(+\Delta t \hat{L}\right), \quad \hat{U}_{\text{back}} = \exp\left(-\frac{\Delta t}{2} \hat{L}\right)
   $$
4. **Field Advancement:** The field is advanced by applying these approximate operators in phase/Fourier space:
   $$
   u \leftarrow \hat{U} \, u
   $$

#### Nonlinear and Source Step
* **Exact Phase Rotation:** If the nonlinearity takes the form $N = i f(|u|^2) u$ (where $f$ is a real-valued function with no spatial derivatives) and the source term $S = 0$, the solver applies an exact, unitary phase rotation:
  $$
  u \to u \exp\left(i f(|u|^2) \Delta t\right)
  $$
* **Runge-Kutta Integration:** For all other cases (e.g., nonlinearities containing derivatives, complex $f$, or active source terms $S \neq 0$), the solver falls back to a classical 4th-order Runge-Kutta (RK4) scheme to integrate the remaining terms.

### 4. Stability, Accuracy & Performance Constraints

Because the linear propagator is a truncated Taylor polynomial rather than a strictly unitary exponential, its stability on the imaginary axis depends entirely on the chosen `order` and the dimensionless parameter $\theta = \Delta t \max|L|$.

* **Stable vs. Unstable Orders:** 
  * *Stable:* Use orders $3, 4, 7, 8, \dots$
  * *Unstable:* Orders $1, 2, 5, 6, 9, 10, \dots$ will artificially amplify the field on the imaginary axis and cause exponential blow-up (triggering a warning).
* **Amplitude Decay:** Even for stable orders, the amplitude may decay slowly (e.g., order 4 exhibits a few percent decay over a run at $\theta \sim 0.5$). To improve accuracy, increase the `order` or reduce $\Delta t$. The truncation error scales as $\mathcal{O}(\Delta t^{\text{order}})$.
* **High-Frequency Instability:** If $\theta$ exceeds the stability limit for the chosen order, high-frequency noise at large $|k|$ or $|x|$ will be amplified. *Remedy:* Reduce $\Delta t$, decrease $N_x$, or shrink the domain size $L_x$.
* **Performance & Precision:** $x$-independent symbols are applied to all runs simultaneously as FFT multipliers, whereas $x$-dependent symbols are applied field-by-field. Computations are performed in double precision internally and cast back to the user-specified `Numerics.precision` afterward.

# Generic 1D PDE Solver

$$\partial_t u = L(u) + N(u) + S(x, t) \quad \text{on a periodic segment}$$


Everything that defines a problem is treated as an **input**:

* **Problem**:
* $L(x, \xi; \text{params})$: linear operator, represented as a SymPy symbol.


* $N(u, x, t; \text{params})$: nonlinear term, written as a rate using SymPy.


* $S(x, t; \text{params})$: source term using SymPy.


* `params`: SymPy symbols (such as $\chi, A_{\text{src}}$, etc.) along with default values.


* `quantization`: `'weyl'` (default) or `'kohn-nirenberg'`.


* `psiop_options`: order, apply backend, apply kwargs, etc.




* **Geometry**: Segment $[-L, L)$ (symmetric, like `psiop.make_grid_1d`), discretized with $N_x$ points.


* **Numerics**: Final time, number of steps, saved frames, precision, etc.


* **Run**: Name, label, initial condition, and a dictionary of parameter values (e.g., `{'chi': 3.0}`). Many runs can be integrated together in a single batch.



The dependence on $\chi$ (or any parameter) is handled as a per-run input: sweeping $\chi$ means providing a list of `Run` instances with different parameter values. Parameters may appear in $L, N, S$, and within the initial condition.

---

## Conventions

* $u(x)$ has Fourier modes proportional to $e^{i k x}$, with $\partial_x \leftrightarrow i\xi$ in the symbol $L$.


* *Example*: $L = -i\xi^2$ corresponds to $\partial_t u = i u_{xx}$.




* $N$ is a rate, e.g., $N = i \chi \vert{}u\vert{}^2 u$. It may contain derivatives of $u$ ($\partial_x u$, complex conjugation, etc.) as well as explicit space and time dependencies $x, t$.


* $S$ is added directly to $\partial_t u$ without any hidden factor of $i$.


* The segment is periodic, utilizing spectral / FFT discretization.



---

## Linear Part: psiop Propagator

For every distinct set of $L$-parameters:

$$L_{\text{kn}} = \text{psiop.weyl\_to\_kn\_symbol}(L) \quad (\text{if quantization == 'weyl'})$$

$$\text{fam} = \text{psiop.PropagatorFamily}(L_{\text{kn}}, [x], \text{order}=\text{order}, \dots)$$

$$\text{prop} = \text{fam.propagator\_for}(+\Delta t/2, +\Delta t, -\Delta t/2)$$

$$u \leftarrow \text{prop.apply}(u, x, k, **\text{apply\_kwargs})$$

The propagator represents the asymptotic expansion of $e^{\Delta t \operatorname{Op}(L)}$: a Taylor series in $\Delta t$ truncated at a given `order`, combined with symbolic Kohn-Nirenberg composition. Because of this, it is **not unitary** and remains accurate only where $\theta = \Delta t \vert{}L\vert{}$ is small on the grid.

Key observed consequences include:

* The Taylor polynomial must be stable on the imaginary axis: use orders $3, 4, 7, 8$, etc.; orders $1, 2, 5, 6, 9, 10$ will amplify the field and blow up after some time, triggering a warning.


* Even for stable orders, the amplitude decays slowly (e.g., order 4 exhibits a few percent decay over a run at $\theta \sim 0.5$); increase the `order` or reduce $\Delta t$ for higher accuracy, noting that error scales as $\mathcal{O}(\Delta t^{\text{order}})$.


* If $\theta = \Delta t \max\vert{}L\vert{}$ exceeds the stability limit for the chosen order, high-frequency noise at large $\vert{}k\vert{}$ or $\vert{}x\vert{}$ is amplified, resulting in a warning: reduce $\Delta t$, $N_x$, or $L_x$.



$x$-independent symbols are applied to all runs simultaneously as FFT multipliers, whereas $x$-dependent symbols are applied one field at a time due to `psiop` limitations. Regarding precision, `psiop` computes in double precision, and fields are subsequently cast back to `Numerics.precision`.

---

## Nonlinear and Source Step

* Exact phase rotation $u \to u \exp(i f(\vert{}u\vert{}^2) \Delta t)$ is used when $N = i f(\vert{}u\vert{}^2) u$ where $f$ is real with no derivatives, and the run's source is zero.


* Classical 4th-order Runge-Kutta (RK4) is used otherwise.

## Imports, symbols and problem specification

In [ ]:
from imports import * 

# ==============================================================================
# 0. SYMBOLS AND PROBLEM SPECIFICATION (ALL INPUTS LIVE HERE)
# ==============================================================================
x_sym, xi_sym, t_sym = symbols('x xi t', real=True)
u_fn = Function('u')(x_sym)          # the unknown u(x)
u_conj = conjugate(u_fn)
I_var = Symbol('I_var', real=True, positive=True)  # stands for |u|^2


@dataclass(frozen=True)
class Geometry:
  """Periodic segment [x_min, x_max) discretised with Nx points."""

  x_min: float = -40.0
  x_max: float = 40.0
  Nx: int = 2048

  @property
  def Lx(self) -> float:
    return self.x_max - self.x_min

  @property
  def dx(self) -> float:
    return self.Lx / self.Nx

  def grid(self, dtype=np.float64):
    x = self.x_min + self.dx * np.arange(self.Nx)
    k = 2.0 * np.pi * fftfreq(self.Nx, d=self.dx)
    return x.astype(dtype), k.astype(dtype)


@dataclass(frozen=True)
class Numerics:
  """Time discretisation and floating-point precision."""

  Lt: float = 7.5              # final time
  Nt: int = 3000               # number of time steps
  n_frames: int = 100          # number of saved frames
  precision: str = 'double'    # 'single' (complex64) or 'double' (complex128)
  workers: int = -1            # FFT threads (-1 = all cores)
  nl_scheme: str = 'auto'      # 'auto' | 'phase' | 'rk4'
  n_jobs: int = 1              # workers over runs (1 = serial, -1 = all usable cores)
  parallel_backend: str = 'process'   # 'process' (fork) or 'thread'

  @property
  def rdtype(self):
    return {'single': np.float32, 'double': np.float64}[self.precision]

  @property
  def cdtype(self):
    return {'single': np.complex64, 'double': np.complex128}[self.precision]

  @property
  def dt(self) -> float:
    return self.Lt / self.Nt



@dataclass
class Problem:
  """d_t u = L(u) + N(u) + S.   `params` are sympy symbols; `defaults` give
  their value when a run does not set them."""

  L: Expr = 0       # symbol L(x, xi; params) of the linear operator
  N: Expr = 0       # nonlinear rate, function of u_fn, x_sym, t_sym, params
  S: Expr = 0       # source, function of x_sym, t_sym, params
  params: tuple = ()
  defaults: dict = field(default_factory=dict)
  # Quantisation of L: 'weyl' (psiop weyl_to_kn_symbol, truncated at
  # weyl_order; exact for polynomial symbols; Hermitian for L = -iH, H real)
  # or 'kohn-nirenberg' (x to the left; e.g. -i(xi+x)^2 then grows ~ exp(2t)).
  # Identical when L has no mixed x*xi term.
  quantization: str = 'weyl'
  weyl_order: int = 4
  # psiop options:
  #   order        Taylor order of the propagator (default 4; use 3,4,7,8,..)
  #   apply_backend 'peetre' (default) or other psiop backends
  #   do_simplify  psiop symbolic simplification (default False)
  #   apply_kwargs passed to prop.apply (default freq_window=None, clamp=inf:
  #                psiop's default Gaussian window would damp the field at
  #                every step)
  #   check        warn about unstable orders / theta = dt*|L| (default True)
  psiop_options: dict = field(default_factory=dict)

  def __post_init__(self):
    self.L, self.N, self.S = (sympify(e) for e in (self.L, self.N, self.S))
    self.params = tuple(self.params)
    if self.quantization not in ('weyl', 'kohn-nirenberg'):
      raise ValueError("quantization must be 'weyl' or 'kohn-nirenberg'.")

  def values(self, run_params: dict) -> list:
    out = []
    for p in self.params:
      if p.name in run_params:
        out.append(float(run_params[p.name]))
      elif p.name in self.defaults:
        out.append(float(self.defaults[p.name]))
      else:
        raise KeyError(
            f"Parameter '{p.name}' has no value and no default in Problem."
        )
    return out


InitialCondition = Union[Callable, Expr, np.ndarray]


@dataclass
class Run:
  """One member of the batch: an initial condition and parameter values."""

  name: str
  label: str
  ic: InitialCondition              # callable(x) | sympy expr(x, params) | array
  params: dict = field(default_factory=dict)


def _build_ic(run: Run, problem: Problem, x, cdtype):
  ic = run.ic
  if isinstance(ic, Basic):
    fn = lambdify((x_sym, *problem.params), ic, modules='numpy')
    vals = fn(x, *problem.values(run.params))
  elif callable(ic):
    vals = ic(x)
  else:
    vals = ic
  vals = np.broadcast_to(np.asarray(vals, dtype=complex), x.shape)
  return vals.astype(cdtype)

## Linear part (psiop propagator) and nonlinear / source compilers

In [ ]:
# ==============================================================================
# 1. LINEAR PART (psiop propagator) AND NONLINEAR / SOURCE COMPILERS
# ==============================================================================
def _import_psiop():
  try:
    from psiop import PropagatorFamily, PseudoDifferentialOperator
  except ImportError as exc:
    raise ImportError(
        'This solver needs the psiop package on the Python path.'
    ) from exc
  return PropagatorFamily, PseudoDifferentialOperator


def psiop_kn_symbol(problem: Problem, L_num):
  """Kohn-Nirenberg symbol of the (numeric-parameter) operator, using psiop's
  own weyl_to_kn_symbol when the problem is Weyl-quantised.  Done ONCE:
  psiop would otherwise redo this sympy conversion at every apply()."""
  _, PDO = _import_psiop()
  if problem.quantization == 'weyl':
    op = PDO(L_num, [x_sym], mode='symbol', quantization='weyl')
    return op.weyl_to_kn_symbol(order=problem.weyl_order)
  return L_num


def _theta_stability_limit(order: int) -> float:
  """Largest theta such that the Taylor polynomial of exp(-i theta) of the
  given order has modulus <= 1 (inf if never exceeded up to theta = 8)."""
  th = np.linspace(1e-3, 8.0, 8000)
  z = -1j * th
  E = sum(z**j / math.factorial(j) for j in range(order + 1))
  bad = np.where(np.abs(E) > 1.0 + 1e-12)[0]
  return float(th[bad[0]]) if bad.size else np.inf


def _check_propagator(problem, L_num, x, k, dt, order, Lt=None):
  """Warn about orders / time steps for which the propagator is unstable."""
  lim = _theta_stability_limit(order)
  if order % 4 in (1, 2):
    warnings.warn(
        f'psiop order={order}: truncated Taylor series of order 1, 2 (mod 4) '
        'amplify the field exponentially on the imaginary axis; use order '
        '3, 4, 7, 8, ...')
    return
  fn = lambdify((x_sym, xi_sym), L_num, 'numpy')
  if L_num.has(x_sym):
    X, K = np.meshgrid(np.asarray(x, float), np.asarray(k, float),
                       indexing='ij')
    theta = dt * float(np.max(np.abs(np.broadcast_to(fn(X, K), X.shape))))
  else:
    theta = dt * float(np.max(np.abs(np.broadcast_to(
        fn(0.0, np.asarray(k, float)), np.shape(k)))))
  if theta > lim:
    hint = ''
    if Lt is not None and np.isfinite(lim):
      nt_stab = int(math.ceil(Lt * theta / dt / (0.9 * lim)))
      nt_acc = int(math.ceil(Lt * theta / dt / 0.5))
      hint = (f'  max|L| = {theta / dt:.0f} on this grid: need Nt >= {nt_stab} '
              f'to be stable, about Nt >= {nt_acc} for accuracy (dt*max|L| '
              f'~ 0.5); or shrink Lx / Nx (max|L| ~ k_max^2 + x_max^2 for a '
              'trap).')
    warnings.warn(
        f'psiop propagator (order {order}): dt*max|L| = {theta:.2f} exceeds '
        f'its stability limit {lim:.2f}; noise at high |k| or |x| will be '
        'amplified.  Reduce dt, Nx or Lx, or raise the order.' + hint)


def make_psiop_steps(problem: Problem, M, x, k, numerics: Numerics):
  """Linear steps u -> exp(+dt/2 Op L) u, exp(+dt Op L) u, exp(-dt/2 Op L) u
  built ONLY from psiop.PropagatorFamily / prop.apply."""
  PropagatorFamily, _ = _import_psiop()

  xd, kd = np.asarray(x, float), np.asarray(k, float)
  dx = float(xd[1] - xd[0])
  if abs(float(xd[0]) + 0.5 * dx * len(xd)) > 1e-8 * dx * len(xd):
    raise ValueError('psiop needs a grid symmetric about 0 (x_min = -x_max), '
                     'like psiop.make_grid_1d.')

  opts = dict(order=4, apply_backend='peetre', do_simplify=False, check=True)
  opts.update({kk: v for kk, v in problem.psiop_options.items()
               if kk != 'apply_kwargs'})
  apply_kwargs = dict(freq_window=None, clamp=np.inf)
  apply_kwargs.update(problem.psiop_options.get('apply_kwargs', {}))

  cd, dt = numerics.cdtype, numerics.dt
  n = M.shape[0]
  names = {s.name for s in problem.L.free_symbols}
  jL = [j for j, p in enumerate(problem.params) if p.name in names]
  groups = {}
  for i in range(n):
    groups.setdefault(tuple(M[i, jL]), []).append(i)

  ops = []
  for idx in groups.values():
    sub = dict(zip(problem.params, M[idx[0]]))
    L_num = problem.L.subs(sub)
    if opts['check']:
      _check_propagator(problem, L_num, xd, kd, dt, opts['order'],
                        Lt=numerics.Lt)
    L_kn = psiop_kn_symbol(problem, L_num)
    fam = PropagatorFamily(L_kn, [x_sym], order=opts['order'],
                           quantization='kohn-nirenberg', mode_composition='kn',
                           apply_backend=opts['apply_backend'],
                           do_simplify=opts['do_simplify'])
    props = [fam.propagator_for(h)[0] for h in (0.5 * dt, dt, -0.5 * dt)]
    ops.append((np.array(idx), props, L_kn.has(x_sym)))

  def make(which):
    def step(U):
      out = np.empty_like(U)
      for idx, props, spatial in ops:
        prop = props[which]
        if not spatial:       # Fourier multiplier: whole batch at once
          out[idx] = prop.apply(U[idx].astype(complex), xd, kd, **apply_kwargs)
        else:                 # x-dependent symbol: one field at a time
          for i in idx:
            out[i] = prop.apply(U[i].astype(complex), xd, kd, **apply_kwargs)
      return out
    return step

  return make(0), make(1), make(2)


def _derivative_order(d: Derivative, x_s) -> int:
  if any(v != x_s for v, _ in d.variable_count):
    raise ValueError(f'Only derivatives in x are supported, got {d}.')
  return sum(n for _, n in d.variable_count)


def compile_nonlinear(problem: Problem):
  """Return (rhs_fn, max_order).

  rhs_fn(u_d0, uc_d0, u_d1, uc_d1, ..., *params, x, t) evaluates N(u), where
  u_dn = d^n u / dx^n (computed spectrally by the integrator).
  Returns (None, 0) if N == 0.
  """
  if problem.N == 0:
    return None, 0

  expr = problem.N.replace(
      Abs, lambda a: sqrt(a * conjugate(a))
  )  # |a| -> sqrt(a a*)
  expr = expr.doit().expand()

  mapping = {u_fn: Symbol('u_d0'), u_conj: Symbol('uc_d0')}
  max_order = 0

  for d in expr.atoms(Derivative):
    n = _derivative_order(d, x_sym)
    if d.expr == u_fn:
      mapping[d] = Symbol(f'u_d{n}')
    elif d.expr == u_conj:
      mapping[d] = Symbol(f'uc_d{n}')
    else:
      raise ValueError(f'Unsupported derivative {d}.')
    max_order = max(max_order, n)

  for c in expr.atoms(conjugate):  # conjugate(Derivative(u, x))
    inner = c.args[0]
    if isinstance(inner, Derivative) and inner.expr == u_fn:
      n = _derivative_order(inner, x_sym)
      mapping[c] = Symbol(f'uc_d{n}')
      max_order = max(max_order, n)

  compiled = expr.xreplace(mapping)
  if compiled.has(u_fn):
    raise ValueError(f'Could not compile nonlinear term: {compiled}')

  dummy = []
  for o in range(max_order + 1):
    dummy += [Symbol(f'u_d{o}'), Symbol(f'uc_d{o}')]

  rhs_fn = lambdify(
      (*dummy, *problem.params, x_sym, t_sym), compiled, modules='numpy'
  )
  return rhs_fn, max_order


def compile_source(problem: Problem):
  """Return fn(x, t, *params) or None if S == 0."""
  if problem.S == 0:
    return None
  return lambdify(
      (x_sym, t_sym, *problem.params), problem.S, modules='numpy'
  )


def extract_phase_law(problem: Problem):
  """If N = i f(|u|^2) u with f real (no derivatives, no explicit x, t),
  return f as a sympy expression in I_var = |u|^2.  Otherwise return None.

  f is what makes the exact phase step possible and defines the nonlinear
  Hamiltonian density  F(I) = int f dI.
  """
  try:
    expr = problem.N.doit()
    if expr == 0:
      return Integer(0)
    if expr.has(Derivative):
      return None
    s = sqrt(I_var)
    th = Symbol('theta', real=True)
    rot = expr.subs(u_conj, exp(-I * th) * s).subs(
        u_fn, exp(I * th) * s
    )
    ref = expr.subs(u_conj, s).subs(u_fn, s)
    if simplify(rot - exp(I * th) * ref) != 0:   # U(1) covariance
      return None
    f = simplify(ref / (I * s))
    if simplify(im(f)) != 0:                         # f must be real
      return None
    if not f.free_symbols <= ({I_var} | set(problem.params)):
      return None                                          # no explicit x, t
    return f
  except Exception:
    return None

## Batched integrator

In [ ]:
# ==============================================================================
# 2. BATCHED INTEGRATOR
# ==============================================================================
def _source_active(problem: Problem, run: Run) -> bool:
  if problem.S == 0:
    return False
  sub = dict(zip(problem.params, problem.values(run.params)))
  s_run = problem.S.subs(sub)
  return not (s_run == 0 or s_run.is_zero is True)


def source_is_linked(run: Run, problem: Problem) -> bool:
  """True if the run explicitly sets a parameter that appears in S."""
  src_names = {s.name for s in problem.S.free_symbols} - {'x', 't'}
  return bool(src_names & set(run.params))


def mask_inactive_sources(results: dict) -> dict:
  """Drop runs that are meant to be driven (they set a source parameter) but
  whose source is identically zero, e.g. A_src = 0.  Runs that never mention
  the source (lin, kerr, ...) are kept.  Use it before plotting."""
  return {
      name: r
      for name, r in results.items()
      if not (
          source_is_linked(r['run'], r['problem'])
          and not _source_active(r['problem'], r['run'])
      )
  }



_FORK_FUNC = None


def _fork_call(arg):
  return _FORK_FUNC(arg)


def usable_cores() -> int:
  """CPU cores this process may really use (respects affinity / container
  limits where the OS reports them; os.cpu_count() can over-report)."""
  try:
    return max(1, len(os.sched_getaffinity(0)))
  except AttributeError:
    return max(1, os.cpu_count() or 1)


def _resolve_jobs(n_jobs, n_tasks, warn=True):
  cores = usable_cores()
  n = cores if n_jobs == -1 else int(n_jobs)
  if warn and n > 1 and cores == 1:
    warnings.warn('n_jobs > 1 requested but only 1 CPU core is usable by this '
                  'process (container / affinity limit?): no speed-up possible.')
  return max(1, min(n, n_tasks))


def _parallel_map(func, args, n_jobs, backend='process'):
  """Map func over args with n_jobs workers.  backend 'process' uses forked
  processes (workers inherit func -- lambdas, sympy objects, psiop operators --
  without pickling; only RESULTS are pickled); 'thread' uses threads (no
  pickling, but limited by the GIL).  Serial fallback if fork is missing."""
  global _FORK_FUNC
  args = list(args)
  n = _resolve_jobs(n_jobs, len(args), warn=False)
  if n == 1:
    return [func(a) for a in args]
  if backend == 'thread':
    from concurrent.futures import ThreadPoolExecutor
    with ThreadPoolExecutor(max_workers=n) as ex:
      return list(ex.map(func, args))
  if backend != 'process':
    raise ValueError("parallel_backend must be 'process' or 'thread'.")
  try:
    ctx = mp.get_context('fork')
  except ValueError:
    warnings.warn("'fork' is not available on this platform (Windows?); "
                  "running SERIALLY.  Use parallel_backend='thread'.")
    return [func(a) for a in args]
  _FORK_FUNC = func
  try:
    with ProcessPoolExecutor(max_workers=n, mp_context=ctx) as ex:
      return list(ex.map(_fork_call, args))
  finally:
    _FORK_FUNC = None


def run_batch(
    problem: Problem,
    runs: Sequence[Run],
    geometry: Geometry = Geometry(),
    numerics: Numerics = Numerics(),
):
  """Integrate all runs. With numerics.n_jobs != 1 the runs are split over
  that many workers (they are independent); each worker integrates its share
  with the serial solver.  A one-line summary reports the measured
  concurrency.  Returns {run.name: result dict}."""
  n = _resolve_jobs(numerics.n_jobs, len(runs))
  if n == 1:
    if numerics.n_jobs not in (1,):
      print(f'[parallel] running serially (n_jobs={numerics.n_jobs}, '
            f'{len(runs)} run(s), {usable_cores()} usable core(s)).')
    return _run_batch_serial(problem, runs, geometry, numerics)

  # avoid oversubscription: split the FFT threads among the workers
  child = dataclasses.replace(
      numerics, n_jobs=1, workers=max(1, usable_cores() // n))
  chunks = [list(range(i, len(runs), n)) for i in range(n)]

  # CPU time (not wall time) of each worker: wall times of workers that
  # time-share a core overlap without any real speed-up.
  cpu_clock = (time.process_time if numerics.parallel_backend == 'process'
               else time.thread_time)

  def work(idxs):
    t0 = cpu_clock()
    res = _run_batch_serial(problem, [runs[i] for i in idxs], geometry, child)
    return (
        {name: {kk: r[kk] for kk in ('label', 'params', 'x', 't', 'frames')}
         for name, r in res.items()},
        os.getpid(), cpu_clock() - t0)

  t_wall = time.time()
  parts = _parallel_map(work, chunks, n, numerics.parallel_backend)
  t_wall = time.time() - t_wall
  merged = {}
  for part, _, _ in parts:
    merged.update(part)
  busy = sum(tt for _, _, tt in parts)
  print(f'[parallel] {len(runs)} runs on {n} {numerics.parallel_backend} '
        f'worker(s), {usable_cores()} usable core(s): wall {t_wall:.1f}s, '
        f'total CPU time {busy:.1f}s -> cores really in use '
        f'{busy / max(t_wall, 1e-9):.2f} (pids {sorted({p for _, p, _ in parts})})')
  out = {}
  for r in runs:                      # original order, full result dicts
    d = merged[r.name]
    d.update(run=r, problem=problem, geometry=geometry, numerics=numerics)
    out[r.name] = d
  return out


def _run_batch_serial(
    problem: Problem,
    runs: Sequence[Run],
    geometry: Geometry = Geometry(),
    numerics: Numerics = Numerics(),
):
  """Integrate all runs simultaneously. Returns {run.name: result dict}."""
  rd, cd = numerics.rdtype, numerics.cdtype
  x, k = geometry.grid(rd)
  Nt, dt = numerics.Nt, rd(numerics.dt)
  workers = numerics.workers

  # --- per-run parameter columns, shape (n_runs, 1) each -------------------
  n_runs = len(runs)
  M = np.array([problem.values(r.params) for r in runs], float).reshape(
      n_runs, len(problem.params)
  )
  cols = [M[:, j : j + 1].astype(rd) for j in range(len(problem.params))]

  fft = lambda a: fft(a, axis=-1, workers=workers)
  ifft = lambda a: ifft(a, axis=-1, workers=workers, overwrite_x=True)

  # --- linear part: psiop propagators ----------------------------------------
  lin_half, lin_full, lin_back = make_psiop_steps(problem, M, x, k, numerics)

  # --- nonlinear / source terms --------------------------------------------
  rhs_fn, max_order = compile_nonlinear(problem)
  src_fn = compile_source(problem)
  f_law = extract_phase_law(problem)

  scheme = numerics.nl_scheme
  if scheme == 'phase' and f_law is None:
    raise ValueError("nl_scheme='phase' needs N = i f(|u|^2) u (f real).")
  use_phase = scheme in ('auto', 'phase') and f_law is not None

  if use_phase:
    active = np.array([_source_active(problem, r) for r in runs])
    if scheme == 'phase' and active.any():
      raise ValueError("nl_scheme='phase' cannot be used with a nonzero source.")
  else:
    active = np.ones(n_runs, bool)

  idx_ph, idx_rk = np.where(~active)[0], np.where(active)[0]
  cols_ph = [c[idx_ph] for c in cols]
  cols_rk = [c[idx_rk] for c in cols]
  f_fn = (
      lambdify((I_var, *problem.params), f_law, modules='numpy')
      if use_phase
      else None
  )

  def rate(u, t, pc):
    """N(u) + S(x, t) for a group of runs."""
    out = 0.0
    if rhs_fn is not None:
      uh = fft(u) if max_order > 0 else None
      args = []
      for o in range(max_order + 1):
        u_d = u if o == 0 else ifft(((1j * k) ** o) * uh)
        args += [u_d, np.conj(u_d)]
      out = out + rhs_fn(*args, *pc, x, rd(t))
    if src_fn is not None:
      out = out + src_fn(x, rd(t), *pc)
    return np.broadcast_to(out, u.shape).astype(cd, copy=False)

  def rk4(u, t, pc):
    h = dt
    k1 = rate(u, t, pc)
    k2 = rate(u + 0.5 * h * k1, t + 0.5 * h, pc)
    k3 = rate(u + 0.5 * h * k2, t + 0.5 * h, pc)
    k4 = rate(u + h * k3, t + h, pc)
    return (u + (h / 6.0) * (k1 + 2 * k2 + 2 * k3 + k4)).astype(cd, copy=False)

  def phase(u, pc):
    f = f_fn(np.abs(u) ** 2, *pc)
    return (u * np.exp(1j * dt * f)).astype(cd, copy=False)

  # --- time loop (Strang splitting) ----------------------------------------
  U = np.stack([_build_ic(r, problem, x, cd) for r in runs])
  save_interval = max(1, Nt // numerics.n_frames)
  frames, t_arr = [U.copy()], [0.0]

  U = lin_half(U)
  for step in range(1, Nt + 1):
    t_c = (step - 1) * numerics.dt
    if idx_ph.size:
      U[idx_ph] = phase(U[idx_ph], cols_ph)
    if idx_rk.size:
      U[idx_rk] = rk4(U[idx_rk], t_c, cols_rk)
    U = lin_full(U) if step < Nt else lin_half(U)

    if step % save_interval == 0 or step == Nt:
      out = U if step == Nt else lin_back(U)
      if not np.isfinite(out).all():
        raise FloatingPointError(
            f'Non-finite field at step {step}: the psiop propagator is '
            'unstable here (see the warnings: order, dt*|L|).')
      frames.append(out.copy())
      t_arr.append(step * numerics.dt)

  frames = np.array(frames)
  return {
      r.name: dict(
          label=r.label,
          params=dict(r.params),
          x=x,
          t=np.array(t_arr),
          frames=frames[:, i, :],
          run=r,
          problem=problem,
          geometry=geometry,
          numerics=numerics,
      )
      for i, r in enumerate(runs)
  }

## Diagnostics: generic Hamiltonian and conserved quantities

In [ ]:
# ==============================================================================
# 3. DIAGNOSTICS: GENERIC HAMILTONIAN AND CONSERVED QUANTITIES
# ==============================================================================
def compute_hamiltonian(r):
  """Hamiltonian for  d_t u = -i dH/du*  (L = -i D with D Hermitian,
  N = i f(|u|^2) u, S arbitrary):

      H0    = int [ u* D u - F(|u|^2) ] dx,   F(I) = int f dI,   D = i Op(L)
      H_ext = 2 Re int i S u* dx

  Op(L) is applied with psiop's apply().  Returns (H0, H0 + H_ext), or
  (nan, nan) if the problem is not of this form or Op(L) is not Hermitian.
  """
  problem, frames, x, t = r['problem'], r['frames'], r['x'], r['t']
  frames = frames.astype(complex)
  x = x.astype(float)
  dx = x[1] - x[0]
  k = 2 * np.pi * fftfreq(len(x), d=dx)
  pv = problem.values(r['params'])
  nan = np.full(len(t), np.nan)

  f_law = extract_phase_law(problem)
  if f_law is None:
    return nan, nan
  F_fn = lambdify(
      (I_var, *problem.params), integrate(f_law, I_var), 'numpy'
  )

  _, PDO = _import_psiop()
  L_num = problem.L.subs(dict(zip(problem.params, pv)))
  op = PDO(psiop_kn_symbol(problem, L_num), [x_sym], mode='symbol',
           quantization='kohn-nirenberg')
  kw = dict(freq_window=None, clamp=np.inf)
  kw.update(problem.psiop_options.get('apply_kwargs', {}))
  Du = 1j * np.stack([op.apply(f, x, k, **kw) for f in frames])

  quad = (np.conj(frames) * Du).sum(-1) * dx            # <u, D u>
  if np.max(np.abs(quad.imag)) > 1e-2 * max(np.max(np.abs(quad.real)), 1e-30):
    return nan, nan                                      # D not Hermitian
  lin_density = np.real(np.conj(frames) * Du)
  nl_density = F_fn(np.abs(frames) ** 2, *pv)
  H0 = (lin_density - nl_density).sum(-1) * dx

  if problem.S != 0:
    S_fn = lambdify((x_sym, t_sym, *problem.params), problem.S, 'numpy')
    S_vals = np.broadcast_to(S_fn(x[None, :], t[:, None], *pv), frames.shape)
    H_ext = (2.0 * np.real(1j * S_vals * np.conj(frames))).sum(-1) * dx
    return H0, H0 + H_ext
  return H0, H0


def compute_diagnostics(r, k_leak=5.0):
  x, frames = r['x'].astype(float), r['frames'].astype(complex)
  dx = x[1] - x[0]
  k = 2 * np.pi * fftfreq(len(x), d=dx)

  uh = fft(frames, axis=-1)
  ux = ifft(1j * k * uh, axis=-1)
  rho = np.abs(frames) ** 2
  H0, H_tot = compute_hamiltonian(r)

  D = dict(
      mass=rho.sum(-1) * dx,
      peak=np.abs(frames).max(-1),
      P=np.imag(np.conj(frames) * ux).sum(-1) * dx,
      H0=H0,
      H=H_tot,
      leak=(np.abs(uh) ** 2)[:, np.abs(k) > k_leak].sum(-1)
      / (np.abs(uh) ** 2).sum(-1),
  )

  for tag, sel in (('p', k > 0), ('m', k < 0)):
    us = ifft(uh * sel, axis=-1)
    rs = np.abs(us) ** 2
    m = rs.sum(-1) * dx
    xc = (x * rs).sum(-1) * dx / np.maximum(m, 1e-30)
    D['m_' + tag] = m
    D['xc_' + tag] = xc
    D['w_' + tag] = np.sqrt(
        (((x - xc[:, None]) ** 2) * rs).sum(-1) * dx / np.maximum(m, 1e-30)
    )
    D['pk_' + tag] = np.abs(us).max(-1)
  return D

## Phase-space analysis: instantaneous manifold extraction

In [ ]:
# ==============================================================================
# 4. INSTANTANEOUS MANIFOLD EXTRACTION (phase-space analysis)
# ==============================================================================
def compute_instantaneous_manifold(
    u_field,
    dx,
    method='madelung',
    threshold=0.05,
    k0=2.0,
    SFT=None,
    Lx=None,
    x_min=None,
    xi_lim=20.0,
    return_coords=False,
):
  N = len(u_field)
  Lx = N * dx if Lx is None else Lx
  x_min = -Lx / 2.0 if x_min is None else x_min
  kx = 2 * np.pi * fftfreq(N, d=dx)
  u_field = np.asarray(u_field, complex)

  def finish(xi_inst, mask):
    if isinstance(xi_lim, (int, float)):
      xi_inst = np.clip(xi_inst, -xi_lim, xi_lim)
    if not return_coords:
      return xi_inst, mask
    x_grid = x_min + dx * np.arange(N)
    return x_grid[mask], xi_inst[mask]

  if method == 'madelung':
    amp = np.abs(u_field)
    mask = amp > (threshold * np.max(amp))
    grad_u = ifft(1j * kx * fft(u_field))
    return finish(np.imag(grad_u / (u_field + 1e-12)), mask)

  if method == 'hilbert':
    u_analytic = hilbert(np.real(u_field))
    amp = np.abs(u_analytic)
    mask = amp > (threshold * np.max(amp))
    phase_hat = fft(np.unwrap(np.angle(u_analytic)))
    return finish(ifft(1j * kx * phase_hat).real, mask)

  if method == 'envelope':
    grad_u = ifft(1j * kx * fft(u_field))
    J = np.imag(np.conj(u_field) * grad_u)
    rho = np.abs(u_field) ** 2
    fringe_period = (2 * np.pi) / (2 * max(abs(k0), 1e-5))
    sigma_pts = (fringe_period / dx) * 0.8
    J_s = gaussian_filter1d(J, sigma=sigma_pts)
    rho_s = gaussian_filter1d(rho, sigma=sigma_pts)
    mask = rho_s > (threshold * np.max(rho_s))
    xi_inst = np.zeros(N)
    xi_inst[mask] = J_s[mask] / rho_s[mask]
    return finish(xi_inst, mask)

  if method == 'reassigned':
    if SFT is None:
      raise ValueError("Method 'reassigned' requires a ShortTimeFFT object.")
    grad_u = ifft(1j * kx * fft(u_field))
    stft_u, stft_ux = SFT.stft(u_field), SFT.stft(grad_u)
    spec_mag = np.abs(stft_u)
    with np.errstate(divide='ignore', invalid='ignore'):
      xi_r = np.imag(stft_ux / stft_u)
    x_stft = SFT.t(N) + x_min

    valid = (spec_mag > threshold * np.max(spec_mag)) & np.isfinite(xi_r)
    if xi_lim is not None:
      if isinstance(xi_lim, (int, float)):
        valid &= np.abs(xi_r) <= xi_lim
      elif len(xi_lim) == 2:
        valid &= (xi_r >= xi_lim[0]) & (xi_r <= xi_lim[1])
    X_mesh = np.tile(x_stft, (spec_mag.shape[0], 1))
    return X_mesh[valid], xi_r[valid]

  raise ValueError(f"Unknown method '{method}'.")


def make_sft(geometry: Geometry, window_length=10.0):
  """Gaussian-window ShortTimeFFT matched to the grid spacing."""
  nwin = int(round(window_length / geometry.dx))
  return ShortTimeFFT(
      gaussian(nwin, std=nwin / 8),
      hop=max(1, nwin // 32),
      fs=1.0 / geometry.dx,
      scale_to='magnitude',
      fft_mode='centered',
  )

## Visualization

In [ ]:
# ==============================================================================
# 5. VISUALIZATION
# ==============================================================================
def _auto_styles(runs):
  colors = plt.rcParams['axes.prop_cycle'].by_key()['color']
  lss = ['-', '--', ':', '-.']
  return {
      n: (colors[i % len(colors)], lss[(i // len(colors)) % len(lss)])
      for i, n in enumerate(runs)
  }


def _param_str(params):
  return ', '.join(f'{k}={v:g}' for k, v in params.items())


def plot_diagnostics(runs, styles=None):
  styles = styles or _auto_styles(runs)
  fig, axs = plt.subplots(2, 3, figsize=(15, 8), sharex=True)

  def metric(ax, key, title, logy=False, relative=False):
    for name, r in runs.items():
      y = r['diag'][key]
      if relative:
        y = y / (y[0] if y[0] != 0 else 1.0) - 1.0
      c, ls = styles[name]
      ax.plot(r['t'], y, color=c, ls=ls, lw=1.6, label=r['label'])
    ax.set_title(title)
    ax.grid(True)
    if logy:
      ax.set_yscale('log')

  metric(axs[0, 0], 'peak', r'Peak $|u|$')
  metric(axs[0, 1], 'w_p', r'RMS Width, $+k$ Beam')
  metric(axs[0, 2], 'xc_p', r'Centroid, $+k$ Beam')
  metric(axs[1, 0], 'mass', 'Total Mass Drift (Relative)', relative=True)
  metric(axs[1, 1], 'leak', r'Spectral Leakage ($|k|>5$)', logy=True)
  metric(axs[1, 2], 'H0', r'Internal $H_0$ Drift (Relative)', relative=True)
  for ax in axs[1]:
    ax.set_xlabel('t')
  axs[0, 0].legend(fontsize=8)
  fig.tight_layout()
  return fig


def plot_reassigned_spectrograms(runs, SFT, times_show, names, xi_lim=10.0):
  """One row per run name in `names`, one column per time in `times_show`."""
  first = runs[names[0]]
  dx = first['x'][1] - first['x'][0]
  x_min = float(first['x'][0])
  Nx = len(first['x'])
  ref = np.abs(SFT.stft(first['frames'][0])).max()
  frame_at = lambda r, tt: int(np.argmin(np.abs(r['t'] - tt)))

  n_cols = len(times_show)
  fig, axs = plt.subplots(
      len(names), n_cols, figsize=(3.5 * n_cols, 4 * len(names)),
      sharex=True, sharey=True, squeeze=False,
  )
  for row, name in enumerate(names):
    r = runs[name]
    for col, tt in enumerate(times_show):
      u = r['frames'][frame_at(r, tt)]
      db = 20 * np.log10(np.maximum(np.abs(SFT.stft(u)) / ref, 1e-6))
      xs = SFT.t(Nx) + x_min
      xi = 2 * np.pi * SFT.f
      ax = axs[row, col]
      ax.imshow(db, aspect='auto', origin='lower', cmap='magma', vmin=-50,
                vmax=0, extent=[xs[0], xs[-1], xi[0], xi[-1]])
      xp, xip = compute_instantaneous_manifold(
          u, dx, method='reassigned', SFT=SFT, Lx=Nx * dx, x_min=x_min,
          xi_lim=xi_lim, threshold=0.03,
      )
      ax.scatter(xp, xip, s=1.2, c='cyan', alpha=0.5)
      ax.set_ylim(-xi_lim, xi_lim)
      ax.set_title(f"{r['label']} | t = {tt:.1f}", fontsize=9)
      if col == 0:
        ax.set_ylabel(r'$\xi$')
      if row == len(names) - 1:
        ax.set_xlabel('x')
  fig.tight_layout()
  return fig


def plot_field_amplitudes(runs, times_show, names, colors=None):
  colors = colors or _auto_styles(names)
  frame_at = lambda r, tt: int(np.argmin(np.abs(r['t'] - tt)))
  n_cols = len(times_show)
  fig, axs = plt.subplots(1, n_cols, figsize=(3.5 * n_cols, 4), sharey=True,
                          squeeze=False)
  axs = axs[0]
  for col, tt in enumerate(times_show):
    for name in names:
      r = runs[name]
      axs[col].plot(r['x'], np.abs(r['frames'][frame_at(r, tt)]),
                    color=colors[name][0], lw=1.2,
                    label=rf"$|u|$, {_param_str(r['params'])}")
    axs[col].set_title(f't = {tt:.1f}')
    axs[col].grid(True)
    axs[col].set_xlabel('x')
  axs[0].legend(fontsize=8)
  fig.tight_layout()
  return fig


def animate_phase_space(run_data, SFT, k0=2.5, xi_lim=10.0,
                        filename='phase_space.gif', n_jobs=1):
  x = run_data['x']
  dx = x[1] - x[0]
  Lx, x_min, Nx = len(x) * dx, float(x[0]), len(x)
  frames, t_arr = run_data['frames'], run_data['t']
  label, pstr = run_data['label'], _param_str(run_data['params'])

  x_stft = SFT.t(Nx) + x_min
  xi_stft = 2.0 * np.pi * SFT.f
  amp_max = 1.1 * np.abs(frames).max()

  fig, (ax_field, ax_spec) = plt.subplots(2, 1, figsize=(10, 8), sharex=True)
  u0 = frames[0]
  (line_re,) = ax_field.plot(x, np.real(u0), 'b-', lw=1.0, label=r'Re$(u)$')
  (line_abs,) = ax_field.plot(x, np.abs(u0), 'r-', lw=1.5, label=r'$|u|$')
  ax_field.set_ylabel('Amplitude')
  ax_field.set_ylim(-amp_max, amp_max)
  ax_field.grid(True)
  ax_field.legend(loc='upper right', fontsize=9)

  spec0 = np.abs(SFT.stft(u0))
  im = ax_spec.imshow(spec0, aspect='auto', origin='lower', cmap='magma',
                      vmin=0, vmax=spec0.max() * 0.7,
                      extent=[x_stft[0], x_stft[-1], xi_stft[0], xi_stft[-1]])
  ax_spec.set_ylim(-xi_lim, xi_lim)
  ax_spec.set_xlabel('x')
  ax_spec.set_ylabel(r'$\xi$')

  def manifolds(u):
    xe, xie = compute_instantaneous_manifold(
        u, dx, method='envelope', threshold=0.04, k0=k0, Lx=Lx, x_min=x_min,
        xi_lim=xi_lim, return_coords=True)
    xr, xir = compute_instantaneous_manifold(
        u, dx, method='reassigned', threshold=0.03, SFT=SFT, Lx=Lx,
        x_min=x_min, xi_lim=xi_lim)
    return np.c_[xe, xie], np.c_[xr, xir]

  # manifolds of every frame are independent: compute them in parallel
  cache = (_parallel_map(lambda i: manifolds(frames[i]),
                         range(len(frames)), n_jobs)
           if n_jobs != 1 else None)
  e0, r0 = cache[0] if cache else manifolds(u0)
  scat_env = ax_spec.scatter(*e0.T, s=3.0, c='cyan', alpha=0.7, label='Envelope')
  scat_rea = ax_spec.scatter(*r0.T, s=2.0, c='deeppink', alpha=0.6,
                             label='Reassigned')
  ax_spec.legend(loc='upper right', fontsize=8, facecolor='black',
                 edgecolor='none', labelcolor='white')

  def update(i):
    u = frames[i]
    line_re.set_data(x, np.real(u))
    line_abs.set_data(x, np.abs(u))
    spec = np.abs(SFT.stft(u))
    im.set_data(spec)
    im.set_clim(vmin=0, vmax=spec.max() * 0.7)
    e, rr = cache[i] if cache else manifolds(u)
    scat_env.set_offsets(e)
    scat_rea.set_offsets(rr)
    ax_field.set_title(f'{label} | t = {t_arr[i]:.2f}')   # label already names the parameters
    return line_re, line_abs, im, scat_env, scat_rea

  anim = FuncAnimation(fig, update, frames=len(frames), interval=50, blit=False)
  anim.save(filename, writer='pillow', fps=20)
  return anim

## Example 1:  d_t u = i u_xx + i chi |u|^2 u

In [ ]:
# ==============================================================================
# 6. EXAMPLE: #  d_t u = i u_xx + i chi |u|^2 u
# ==============================================================================
# ---- parameters (symbols) -------------------------------------------------
chi = Symbol('chi', real=True)
A_src, x0_src, t0_src, w0_src, tau0_src = symbols(
  'A_src x0_src t0_src w0_src tau0_src', real=True)

# ---- the three operators: d_t u = L(u) + N(u) + S -------------------------
problem = Problem(
  L=-I * xi_sym**2,                                  # d_t u = i u_xx
  N=I * chi * Abs(u_fn) ** 2 * u_fn,              # Kerr, chi-dependent
  S=I * A_src * exp(
      -((x_sym - x0_src) ** 2) / w0_src**2
      - ((t_sym - t0_src) ** 2) / tau0_src**2),         # Gaussian pulse
  params=(chi, A_src, x0_src, t0_src, w0_src, tau0_src),
  defaults=dict(chi=0.0, A_src=0.0, x0_src=0.0, t0_src=0.0,
                w0_src=1.0, tau0_src=1.0),
  quantization='weyl',
  # order 3, 4, 7, 8 are stable; the error decreases with the order
  psiop_options=dict(order=4),
)
# Trapped variant (x-dependent L): use  L=-I*(xi_sym**2 + x_sym**2)  with
# Geometry(-20, 20, 256) and a much smaller dt (see the stability warnings).

# ---- geometry and numerical precision --------------------------------------
# psiop propagator: dt*k_max^2 must stay below the stability limit of the
# order (k_max = pi/dx), so Nx = 512 here instead of 2048.
geometry = Geometry(x_min=-40.0, x_max=40.0, Nx=512)
numerics = Numerics(Lt=7.5, Nt=3000, n_frames=100, precision='single',
                  workers=-1, nl_scheme='auto',
                  n_jobs=-1)           # runs in parallel on all usable cores

# ---- initial conditions ----------------------------------------------------
W0, K0, X1, X2 = 3.0, 2.5, -20.0, 20.0
ic_two = lambda x: (np.exp(-((x - X1) ** 2) / W0**2 + 1j * K0 * x)
                  + np.exp(-((x - X2) ** 2) / W0**2 - 1j * K0 * x))
ic_single = lambda x: np.exp(-((x - X1) ** 2) / W0**2 + 1j * K0 * x)
ic_soliton = 1 / cosh(x_sym)         # a sympy expression also works

# ---- the chi dependence: one Run per value of chi --------------------------
chi_values = {'lin': 0.0, 'weak': 0.3, 'kerr': 3.0}
runs = [
  Run(name, f'two beams, chi={c:g}', ic_two, {'chi': c})
  for name, c in chi_values.items()
]
runs += [
  Run('single', 'one beam, chi=3 (SPM only)', ic_single, {'chi': 3.0}),
  Run('driven', 'one beam, chi=3 + pulse S', ic_single,
      {'chi': 3.0, 'A_src': 1.0, 'x0_src': 0.0, 't0_src': 3.5,
       'w0_src': 2.0, 'tau0_src': 1.0}),          # A_src=0 switches S off
  # sech(x) is the exact soliton of this equation for chi = 2
  Run('soliton', 'soliton test, chi=2', ic_soliton, {'chi': 2.0}),
]

print('Starting batched simulation...')
t0 = time.time()
results = run_batch(problem, runs, geometry, numerics)
print(f'Integration complete in {time.time() - t0:.2f}s.')

sol = results.pop('soliton')
pk = np.abs(sol['frames']).max(axis=1)
print(f'Soliton peak drift: {np.abs(pk / pk[0] - 1).max():.2e}')

for r in results.values():
    r['diag'] = compute_diagnostics(r)

SFT = make_sft(geometry)
plot_diagnostics(mask_inactive_sources(results))  # hides driven run if A_src=0
plt.show()

times_show = [0.1, 2.0, 4.0, 6.0, 8.0]
plot_reassigned_spectrograms(results, SFT, times_show, names=['lin', 'kerr'])
plt.show()
plot_field_amplitudes(results, times_show, names=['lin', 'kerr'])
plt.show()

print('Generating phase-space animation...')
animate_phase_space(results['kerr'], SFT, k0=K0, n_jobs=-1,
                  filename='nlse_colliding_beams_phase_space.gif')
plt.show()

## Example 2 : d_t u = i u_xx - i Omega^2 x^2 u + i chi |u|^2 u

In [ ]:
# ==============================================================================
# 7. EXAMPLE: d_t u = i u_xx - i Omega^2 x^2 u + i chi |u|^2 u
# ==============================================================================
# ---- parameters ------------------------------------------------------------
chi, Omega = symbols('chi Omega', real=True)

# ---- d_t u = i u_xx - i Omega^2 x^2 u + i chi |u|^2 u ----------------------
problem = Problem(
  L=-I * xi_sym**2 - I * Omega**2 * x_sym**2,     # harmonic oscillator symbol
  N=I * chi * Abs(u_fn) ** 2 * u_fn,              # Kerr
  params=(chi, Omega),
  defaults=dict(chi=0.0, Omega=1.0),
  quantization='weyl',        # no mixed x*xi term, so same as 'kohn-nirenberg'
  psiop_options=dict(order=3),                          # stable orders: 3, 4, 7, 8
)

# ---- geometry / numerics: dt * max|L| <= ~0.5 -------------------------------
# max|L| ~ k_max^2 + Omega^2 x_max^2 = 400 + 400 = 800
geometry = Geometry(x_min=-20.0, x_max=20.0, Nx=256)
dt = 6.25e-4
Lt = 2*3.1416                      # one period for Omega = 1 (period = pi/Omega)
numerics = Numerics(Lt=Lt, Nt=int(round(Lt / dt)), n_frames=100, precision='double',
                      workers=-1, nl_scheme='auto', n_jobs=-1)  

# ---- beams inside the trap (orbit radius about 10) -------------------------
W0, K0, X1, X2 = 3.0, 2.5, -10.0, 10.0
ic_two = lambda x: (np.exp(-((x - X1) ** 2) / W0**2 + 1j * K0 * x)
                  + np.exp(-((x - X2) ** 2) / W0**2 - 1j * K0 * x))
runs = [Run('lin',  'two beams, chi=0', ic_two, {'chi': 0.0}),
      Run('kerr', 'two beams, chi=3', ic_two, {'chi': 3.0})]

print('Starting batched simulation...')
t0 = time.time()
results = run_batch(problem, runs, geometry, numerics)
print(f'Integration complete in {time.time() - t0:.2f}s.')

for r in results.values():
    r['diag'] = compute_diagnostics(r)

SFT = make_sft(geometry)
plot_diagnostics(mask_inactive_sources(results))  # hides driven run if A_src=0
plt.show()

times_show = [0.1, 2.0, 4.0, 6.0, 8.0]
plot_reassigned_spectrograms(results, SFT, times_show, names=['lin', 'kerr'])
plt.show()
plot_field_amplitudes(results, times_show, names=['lin', 'kerr'])
plt.show()

print('Generating phase-space animation...')
animate_phase_space(results['kerr'], SFT, k0=K0,
                  filename='nlho_colliding_beams_phase_space.gif')
plt.show()

In [ ]:
animate_phase_space(results['lin'], SFT, k0=K0,
                  filename='lho_colliding_beams_phase_space.gif')